Logistic Regression(binary classifier): Very similar set up to linear regression, however now we are going to take the weighted sum from the parameter column vector and the row vector of the feature instance and turn it into a probability using the sigmoid function, and with a threshold we set we will determine whether it falls into the y=1 or y=0 category. So, we are trying to find the probability that y=1 given our data(feature instances). The probability that y=0 is equal to (1-probability y=1). So the differences between linear regression and logistic regression are: the hypothesis is now 1/1+e^(-weighted sum), the cost function is the logged bernoulli function so the cost is meausred by how close the hypothesis was to the true classification


In [46]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression


Here I will create the function to train the model to find the parameters that give the smallest cost using gradient descent. So it returns a column vector of parameters

In [47]:
def sigmoid(res):
    return 1/(1+np.exp(-res)) #returns the weighted sum squished into a probability
def fit_model_iteration(X,y,alpha=0.1,epoches=10000):
    gradient_vector = np.zeros_like(X[0],dtype=float) #holds the gradient vector for the partials so we can minimize the cost function
    parameters = np.zeros_like(X[0],dtype=float) #this is the parameter vector that will be returned in the end (fitting this)
    m = len(X) #number of rows we have in the data
    for e in range(epoches):
        #go through all the parameters to get the partials of each one to store as the gradients
        for j in range(len(parameters)):
            #getting running sum of error on predictions for each gradient scaled by their corresponding feature
            current_partial_sum = 0
            for i in range(len(X)):
                pred_prob = sigmoid(X[i]@parameters)
                current_partial_sum += (pred_prob-y[i])*X[i][j] #scaled by the corresponding feature of the current parameter
            gradient_vector[j] = current_partial_sum / m #stores the current partial derivative we found 
        #now we update the parameters after a round of gradient descent
        parameters = parameters - alpha*gradient_vector
    return parameters

In [49]:
def fit_model_vectorized(X,y,alpha=0.1,epoches=10000):
    gradients = np.zeros_like(X[0],dtype=float)
    parameters = np.zeros_like(X[0],dtype=float)
    m = len(X)
    #going to compute the gradients and update the parameters using vectorized operations
    for i in range(epoches):
        probas = sigmoid(X@parameters) #vector of probabilites from weighted sums using current parameters
        gradients = X.T @ (probas-y)/m
        parameters -= alpha*gradients
    return parameters


In [48]:
# Fake test data: 2 features, binary labels, no preprocessing needed
np.random.seed(42)
m = 60  # number of instances (kept small so the triple loop runs fast)

# two blobs: class 0 centered around (-1, -1), class 1 centered around (1, 1)
X0 = np.random.randn(m // 2, 2) * 0.8 + np.array([-1, -1])
X1 = np.random.randn(m // 2, 2) * 0.8 + np.array([1, 1])
X_features = np.vstack([X0, X1])
y = np.concatenate([np.zeros(m // 2), np.ones(m // 2)])

# add the bias column of 1s as the first column (x0 = 1)
X = np.c_[np.ones(m), X_features]

# shuffle so the classes are mixed
idx = np.random.permutation(m)
X, y = X[idx], y[idx]

print(X.shape, y.shape)
print(X[:5])
print(y[:5])


(60, 3) (60,)
[[ 1.          1.73232169  1.26300089]
 [ 1.         -1.5415376  -0.51065897]
 [ 1.          1.04818417  2.97059369]
 [ 1.          0.26337025 -0.38605222]
 [ 1.          1.07766204  1.77491599]]
[1. 0. 1. 0. 1.]


In [50]:
params = fit_model_vectorized(X,y)

In [51]:
#returns a vector of predictions using the parameters we got from the fitted model
#we first get the scores from the weighted sums and then squish into probabilities with sigmoid function and compare to our threshold
def predict(X,parameters,threshold=0.5):
    scores = X@parameters
    probabilites = np.array([sigmoid(scores[i]) for i in range(len(scores))])
    predictions_mask = probabilites >= threshold #boolean mask
    predictions = np.astype(predictions_mask, int) #converts to 0 or 1 
    return predictions
def score(X,y,eps=1e-15):
    probs = np.clip(X,eps,1-eps)
    curr_cost = 0
    for i in range(len(X)):
        curr_cost += (-y[i]*np.log(probs[i])) - ((1-y[i])*np.log(1-probs[i]))
    return curr_cost / len(probs)

In [52]:
predictions = predict(X,params)
scores = score(predictions,y)

In [53]:
predictions[:10],y[:10]

(array([1, 0, 1, 0, 1, 1, 0, 0, 1, 1]),
 array([1., 0., 1., 0., 1., 1., 0., 0., 1., 1.]))

In [54]:
log_model = LogisticRegression()
trained = log_model.fit(X,y)


In [55]:
predictions = trained.predict(X)

np.float64(0.0)